In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _prompt
import _mapping
import _util
from _intervention import get_label_probability, forward_with_cache, get_logit_lens, get_label_probability_from_logits

In [3]:
model_type = "GPT-OSS_stepwise" # GPT-OSS or R1

if "GPT-OSS" in model_type:
    model, tokenizer = _util.load_OSS()
elif "R1" in model_type:
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
# Print all modules of the model
print("Model modules:")
for name, module in model.named_modules():
    print(f"{name}: {type(module).__name__}")

Model modules:
: GptOssForCausalLM
model: GptOssModel
model.embed_tokens: Embedding
model.layers: ModuleList
model.layers.0: GptOssDecoderLayer
model.layers.0.self_attn: GptOssAttention
model.layers.0.self_attn.q_proj: Linear
model.layers.0.self_attn.k_proj: Linear
model.layers.0.self_attn.v_proj: Linear
model.layers.0.self_attn.o_proj: Linear
model.layers.0.mlp: GptOssMLP
model.layers.0.mlp.router: GptOssTopKRouter
model.layers.0.mlp.experts: GptOssExperts
model.layers.0.input_layernorm: GptOssRMSNorm
model.layers.0.post_attention_layernorm: GptOssRMSNorm
model.layers.1: GptOssDecoderLayer
model.layers.1.self_attn: GptOssAttention
model.layers.1.self_attn.q_proj: Linear
model.layers.1.self_attn.k_proj: Linear
model.layers.1.self_attn.v_proj: Linear
model.layers.1.self_attn.o_proj: Linear
model.layers.1.mlp: GptOssMLP
model.layers.1.mlp.router: GptOssTopKRouter
model.layers.1.mlp.experts: GptOssExperts
model.layers.1.input_layernorm: GptOssRMSNorm
model.layers.1.post_attention_layernor

In [5]:
prompt_type = "h_pre_result" # empty or pre_result or pre_sum
intervention_loc = "final_sum" # restatement or reasoning or restatement_and_reasoning

# Load the divided prompts dataset
if 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[1:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} divided prompts")

loaded 256 divided prompts


In [6]:
if 'h' in prompt_type:
    if model_type == "GPT-OSS":
        intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
    elif model_type == "R1":
        intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
else:
    if model_type == "GPT-OSS":
        intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
    elif model_type == "R1":
        intervention_ids_dict = _mapping.intervene_ids_R1_3_digit

if type(intervention_loc) == str:
    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]

intervention_ids = [25]
print(intervention_ids)

[25]


In [7]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['intervention_prompt', 'layer', 'factual_label_probability', 'counterfactual_label_probability']
tok_pos=-1
filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}/logit_lens", f"{prompt_type}_{intervention_loc}_{tok_pos}.csv", header, overwrite=False)

batch_size = 24

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    factual_labels = batch_rows['base_sum']
    counterfactual_labels = batch_rows['source_sum']
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to('cpu')
    
    base_labels = tokenizer([str(base_sum) for base_sum in factual_labels], add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right")["input_ids"].to('cpu')
    base_labels_mask = base_labels != tokenizer.pad_token_id
    source_labels = tokenizer([str(source_sum) for source_sum in counterfactual_labels], add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right")["input_ids"].to('cpu')
    source_labels_mask = source_labels != tokenizer.pad_token_id

    output, activations = forward_with_cache(model, tokens['input_ids'], pre_hook=False)
    for layer in range(len(model.model.layers)):
        activation = activations[f'model.layers.{layer}']
        logits = get_logit_lens(model, activation, tok_pos=tok_pos)
        base_labels_probs = get_label_probability_from_logits(logits, base_labels)
        source_labels_probs = get_label_probability_from_logits(logits, source_labels)
    
        # Process each generated text in the batch
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            _util.write_to_csv(filepath, row.to_list() + [intervention_prompts[j], layer, base_labels_probs[j].item(), source_labels_probs[j].item()])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [01:22<00:00,  7.51s/it]
